# Answer questions with the Chinook database

Chinook is a sample music-store database with artists, albums, tracks, customers,
employees, and sales. Run the cells in order to inspect its tables, generate SQL
from a question, execute that SQL, and use the results to produce an answer.
This notebook uses Cornell's gateway and the same configuration as the earlier
model-call workbook. No additional API key or database server is needed.

Here, SQL retrieves rows for the answer instead of using embeddings to retrieve
text chunks. The database is included in `assignments/02-structured-data-rag/data/Chinook.db` and is
opened read-only. You can rerun the notebook without changing its records.

## What to watch for

Read the generated SQL before executing the next cell. Model-written SQL can
still be incorrect even when it runs successfully. The question "What are the
three biggest hits?" is deliberately open to interpretation: does "biggest"
mean units sold, revenue, or something else? Compare the chosen query with what
you intended.

The final examples ask about sales support employees and trending genres.
Treat these as a discussion of the limits of the evidence: sales records do not
establish an employee's awareness of trends or justify a promotion. A better
follow-up is to ask a measurable sales question with a clearly defined date
range. The included invoices span 2021 through 2025, so specify a year instead
of relying on "last year" when repeating the example in the future.

The cleanup function removes common SQL formatting in Python. It does not check
whether a query answers the question correctly. The read-only database connection
prevents writes to the supplied database.


In [1]:
# These helpers display formatted output in the notebook.
# We use Markdown below to render the model's answers.
from IPython.display import display, Markdown, HTML

In [2]:
# SQLDatabase lets LangChain inspect tables and run SQL queries.
# pandas displays query results as a table; Path helps locate our database file.
from langchain_community.utilities import SQLDatabase
from IPython.display import display
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine

# SQLite stores this entire sample database in one file, so no server is needed.
# The first path works from this notebook's folder; the second works from the repo root.
db_path = Path("data/Chinook.db")
if not db_path.is_file():
    db_path = Path("assignments/02-structured-data-rag/data/Chinook.db")
if not db_path.is_file():
    raise FileNotFoundError("Open this notebook from the repository root or assignments/02-structured-data-rag folder.")

# This connection address identifies SQLite and the full path to the file.
# mode=ro opens it read-only, so generated queries cannot change its records.
db_uri = f"sqlite:///file:{db_path.resolve().as_posix()}?mode=ro&uri=true"
db = SQLDatabase.from_uri(db_uri)

# The dialect tells us which variety of SQL to use. The table names tell us
# what data we can query. None of these operations calls a language model.
display(db.dialect)
display(db.get_usable_table_names())

# An engine manages database connections. pandas uses this one to execute SQL.
engine = create_engine(db_uri)

# Start with a query we wrote ourselves: show up to 10 rows from Artist.
query = "SELECT * FROM Artist LIMIT 10;"
df = pd.read_sql_query(query, engine)
display(df)


'sqlite'

['Album',
 'Artist',
 'Customer',
 'Employee',
 'Genre',
 'Invoice',
 'InvoiceLine',
 'MediaType',
 'Playlist',
 'PlaylistTrack',
 'Track']

,ArtistId,Name
0,1,AC/DC
1,2,Accept
2,3,Aerosmith
3,4,Alanis Morissette
4,5,Alice In Chains
5,6,Antônio Carlos Jobim
6,7,Apocalyptica
7,8,Audioslave
8,9,BackBeat
9,10,Billy Cobham


In [3]:
# ChatOpenAI is LangChain's interface to the model.
# It reads OPENAI_API_KEY and OPENAI_BASE_URL from the Codespace environment.
from langchain_openai import ChatOpenAI
from os import environ

# This creates the model interface. A later .invoke(...) call sends the request.
llm = ChatOpenAI(model="openai.gpt-4o")


In [4]:
from langchain.chains import create_sql_query_chain

# A chain connects processing steps. This one sends the question, table
# definitions, and sample rows to the model so it can write SQLite SQL.
# It returns SQL text; it does not execute the generated query.
chain = create_sql_query_chain(llm, db)
response = chain.invoke({"question": "What are the three biggest hits?"})

# response is just our variable name. Here it contains the generated SQL.
# What did the model decide "biggest hits" means: sales, revenue, or something else?
display(Markdown(response))


```sql
SQLQuery: 
SELECT "Name", "Milliseconds" 
FROM "Track" 
ORDER BY "Milliseconds" DESC 
LIMIT 3;
```

In [5]:
# Inspect the prompt template chosen by LangChain for this SQL database.
# Look for the SQL instructions and the input/table_info placeholders.
# The chain fills those placeholders when invoked. This cell makes no model call.
chain.get_prompts()[0].pretty_print()


You are a SQLite expert. Given an input question, first create a syntactically correct SQLite query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most 5 results using the LIMIT clause as per SQLite. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in double quotes (") to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use date('now') function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: SQL Query to run
SQLResult: Result

In [6]:
# re provides text-pattern matching. This helper handles response formatting;
# it does not verify SQL correctness or decide whether the query answers the question.
import re

def clean_sql_query(response):
    # Remove surrounding whitespace, then extract SQL from a code block if present.
    query = response.strip()
    fenced_sql = re.search(r"```(?:sql)?\s*(.*?)```", query, re.DOTALL | re.IGNORECASE)
    if fenced_sql:
        query = fenced_sql.group(1).strip()

    # The chain can stop before a closing code fence is generated.
    # Also remove a leading SQLQuery: label, which SQLite would not understand.
    query = re.sub(r"^```(?:sql)?\s*", "", query, flags=re.IGNORECASE)
    query = re.sub(r"^SQLQuery:\s*", "", query, flags=re.IGNORECASE)

    # Skip introductory prose and find a SELECT query or a WITH clause.
    # WITH names a temporary query result that the rest of the SQL can reference.
    sql_start = re.search(r"^\s*(?:SELECT|WITH)\b", query, re.MULTILINE | re.IGNORECASE)
    if sql_start is None:
        raise ValueError(f"No SELECT or WITH query found in model output: {response!r}")
    return query[sql_start.start():].strip()


In [7]:
# Clean the SQL we generated earlier and inspect it before running it.
# This is ordinary Python text processing, with no additional model call.
clean_response = clean_sql_query(response)
print(clean_response)


SELECT "Name", "Milliseconds" 
FROM "Track" 
ORDER BY "Milliseconds" DESC 
LIMIT 3;


In [8]:
# Now execute the query against SQLite. The database supplies the actual records.
# In this example, db.run returns a string showing the result rows as tuples.
# Check the query's joins, ordering, and limits before interpreting those rows.
db.run(clean_response)


"[('Occupation / Precipice', 5286953), ('Through a Looking Glass', 5088838), ('Greetings from Earth, Pt. 1', 2960293)]"

In [9]:
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool

# Wrap SQL execution as a LangChain tool so we can connect it to other steps.
# A tool here is called by our pipeline; the model is not choosing when to use it.
execute_query = QuerySQLDataBaseTool(db=db)
write_query = create_sql_query_chain(llm, db)

# The | operator passes each step's output to the next step:
# question dictionary -> SQL text -> cleaned SQL -> database result.
# We now assign chain to this larger pipeline, replacing the earlier SQL-only chain.
chain = write_query | clean_sql_query | execute_query

# This returns the query result, before any model-written explanation.
# The tool returns SQL errors as text too, so inspect the output for errors.
chain.invoke({"question": "How many employees are there"})


'[(8,)]'

In [10]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# The answering model needs the original question, the SQL, and its result.
# The names inside braces are placeholders filled from a dictionary at runtime.
answer_prompt = PromptTemplate.from_template(
    """Given the following user question, corresponding SQL query, and SQL result, answer the user question.

Question: {question}
SQL Query: {query}
SQL Result: {result}
Answer: """
)

# Keep the original question as we add information to that dictionary:
# {question} -> {question, query} -> {question, query, result}.
# assign adds a field; itemgetter("query") selects just the SQL for execution.
chain = (
    RunnablePassthrough.assign(query=write_query | clean_sql_query)
    .assign(
        result=itemgetter("query") | execute_query
    )
    | answer_prompt      # Fill the template with the question, SQL, and result.
    | llm                # Ask the model to explain the database result.
    | StrOutputParser()  # Extract the model message's text as a Python string.
)

# Building this pipeline does not run it. Each invocation below makes two
# model calls: one to write SQL and one to turn the result into an answer.


In [11]:
# Run the full pipeline and compare its answer with the raw employee count above.
# response now holds the final answer, because chain now includes the answering step.
response = chain.invoke({"question": "How many employees are there?"})

display(Markdown(response))


There are 8 employees.

In [12]:
# The pipeline writes a fresh query each time; it does not reuse the earlier SQL.
# There is no conversation history here. This question is answered independently.
response = chain.invoke({"question": "What are the three biggest hits?"})

# Did the model use the same meaning of "biggest hits" as before?
# Try specifying "tracks ranked by total units sold across all invoices" instead.
display(Markdown(response))


The three biggest hits are:

1. **The Trooper** with 5 sales.
2. **Untitled** with 4 sales.
3. **The Number Of The Beast** with 4 sales.

In [13]:
# This question goes beyond what sales records can establish.
# A high sales count does not prove awareness of trends or merit a promotion.
# Also ask what date range "last year" should mean for this historical dataset.
response = chain.invoke({"question": "Which of our sales support agents should be promoted based on their awareness of trending genres, trending genres can be classified based on the last year of sales?"})

# Check whether the answer separates observed sales from unsupported conclusions.
display(Markdown(response))


Based on their awareness of trending genres, **Jane Peacock** should be promoted. She has the highest sales (304 units) in the trending genre over the past year, demonstrating her strong understanding of customer preferences and market trends.

In [14]:
# Return to a SQL-only chain to examine how a similar question becomes a query.
# This generates new SQL; it does not reveal the exact SQL used in the previous cell.
test_chain = create_sql_query_chain(llm, db)
response = test_chain.invoke({"question": "Which of our sales support agents should be promoted based on their awareness of trending genres, can be classified based on the last year of sales??"})

# Inspect the date filter, joins, and ranking rule. What can those actually measure?
# No SQL is executed by this cell.
display(Markdown(response))


To determine which sales support agent should be promoted based on their awareness of trending genres, we would need to analyze the sales (in terms of invoices) handled by each agent in the last year and correlate that with the genres of the tracks sold.

However, the provided data does not include direct information about which tracks (and therefore genres) were sold per agent, as the "Invoice" table links sales to customers, not directly to employees. To achieve this, we could join the "Invoice," "Customer," "Employee," and "Track" tables through intermediate relations to analyze genre sales per agent.

Here’s the SQL query for identifying the top-performing sales support agent based on the genres of tracks sold in the last year (assuming year 2022):

SQLQuery:
```sql
SELECT 
    e."FirstName" || ' ' || e."LastName" AS "AgentName",
    g."Name" AS "Genre",
    COUNT(il."InvoiceLineId") AS "SalesCount"
FROM 
    "Invoice" i
JOIN 
    "Customer" c ON i."CustomerId" = c."CustomerId"
JOIN 
    "Employee" e ON c."SupportRepId" = e."EmployeeId"
JOIN 
    "InvoiceLine" il ON i."InvoiceId" = il."InvoiceId"
JOIN 
    "Track" t ON il."TrackId" = t."TrackId"
JOIN 
    "Genre" g ON t."GenreId" = g."GenreId"
WHERE 
    i."InvoiceDate" >= date('now', '-1 year')
GROUP BY 
    e."EmployeeId", g."GenreId"
ORDER BY 
    "SalesCount" DESC
LIMIT 5;
```